# INSTRUCTOR GUIDE: Week 4 Session 1
The pandas analytical workflow (Greenleaf Market)

**Not for students.** Solutions are filled in. Timed run of show, talking points, and expected numbers are at the top.

---

## Timed run of show (2 hours)

| Clock | Block | What you do |
|---|---|---|
| 0:00-0:10 | Check-in + mission | Put the mission question on the board. This is a fictional neighbourhood shop. Walk the 9-step workflow. |
| 0:10-0:30 | Concept | What one row means in each table (grain). Why two tables. Merge keys. Common mistakes: chaining without inspecting; merge without checking keys and grain. |
| 0:30-1:00 | Live coding Parts 1-5 | Load both files, inspect, missing values, the quantity 0, duplicate row, filter/transform with `.loc`, then merge and discover P99 and P0X. |
| 1:00-1:10 | Break | |
| 1:10-1:45 | Guided practice Parts 6-8 | Drop or flag bad rows with a written justification. Compute line_total. Groupby category. Two charts. Findings plus caveats (Household has only 1 usable row). |
| 1:45-1:55 | Challenge | Top product by revenue, with count and a caveat. |
| 1:55-2:00 | Recap + assign | Recap the workflow. Homework: Your turns + findings paragraph. Kick off the EDA project with **open dataset choice**. |

## Talking points
- **Mission first.** Every step should help us trust (or question) category revenue.
- **Grain / keys.** `products`: one row is one product. `orders`: one row is one order line. The key is `product_id`. After a left merge, row count should stay 111 (including the duplicate) or 110 if you dropped the duplicate first. Unmatched keys show up as missing product_name.
- **Inspect before you clean.** Students will want to drop NaNs immediately. Make them list problems first.
- **P12 has no unit_price.** Those 5 orders cannot become revenue until we fill or drop them. Say so out loud.
- **Household after cleaning has 1 row.** Do not let anyone crown a "worst category" from one row.
- **Correlation is not the topic today**, but the same honesty applies: a total is not a story without counts and caveats.
- **EDA project.** Dataset choice is open. Greenleaf is one option, not a requirement.

## Expected numbers (from the files in this folder)
- `products.shape` -> (12, 5). One missing `unit_price` (P12 Detergent).
- `orders.shape` -> (111, 6) including 1 exact duplicate. After `drop_duplicates()`, 110 rows.
- Missing `quantity`: 4. Missing `payment`: 3. `quantity == 0`: 1 row.
- Bad keys: **ORD018 / P99** and **ORD064 / P0X** (not in the products list).
- Unique `product_id` in orders: 14 (12 real + 2 typos).
- After left merge on 110 cleaned orders: 110 rows; 2 unmatched; 4 missing quantity; 5 P12 rows with missing price (7 missing unit_price if you also count the 2 unmatched).
- Usable revenue rows (non-missing line_total): **99**. Total revenue: **497,000**.
- Category totals (usable rows): Staples **438,500** (41 rows), Drinks **38,600** (33), Snacks **18,150** (24), Household **1,750** (1 row).
- Cashiers in the raw file (with the duplicate): Bisi 41, Tunde 37, Amaka 33.

## Common mistakes to watch for
1. Merging on the wrong key, or on `product_name` (not present in orders).
2. Using an inner merge and not noticing that two typo orders silently disappeared.
3. Computing `quantity * unit_price` before the merge (there is no price on orders).
4. Dropping all missing values in one line without saying what was lost.
5. Treating Household as a firm last-place category (n=1).
6. f-strings (course rule: use `print` with commas or concatenation).

---

# Student-facing content begins below (solutions filled in)


# Week 4 Session 1: The pandas analytical workflow
**Forge Academy | Data Science Foundations**
*Week 4 Session 1 — Working with Pandas for analytical workflows*

Today we practise a full analytical workflow on a small, concrete business question.

**Mission question**

> Which product categories bring in the most revenue at Greenleaf Market, and which orders need cleaning before we can trust that answer?

Greenleaf Market is a **fictional** neighbourhood shop. The two CSV files were built for this class. They are small on purpose, and they contain a few realistic problems.

Every step below serves the mission: load, inspect, clean, transform, merge, group, plot, then conclude with honest caveats.

**By the end of this session you can:**
1. Follow a pandas analytical workflow from a question to a conclusion.
2. Load and inspect more than one CSV.
3. Find missing values, duplicates, and odd values, and decide what to do with them.
4. Filter rows with boolean masks and create new columns with `.loc`.
5. Merge two tables on a shared key, after checking what one row means in each table.
6. Use `groupby` to compare groups, draw 1 to 2 simple charts, and state findings carefully.

**Setup:** keep these files in the **same folder** as this notebook:
- `products.csv`
- `orders.csv`

Open the notebook in VS Code, select your Python kernel, and run cells from top to bottom. Cells marked **Your turn** are for you to write code.

**Session map**
| Part | Topic |
|---|---|
| 0 | Mission and the workflow |
| 1 | Load the data |
| 2 | Inspect |
| 3 | Missing data, duplicates, and odd values |
| 4 | Filter and transform |
| 5 | Merge products with orders |
| 6 | Groupby: answer the mission |
| 7 | Two light charts |
| 8 | Findings and caveats |
| 9 | Challenge, recap, homework, EDA project kickoff |

## 0. Mission and the workflow

A shopkeeper can look at a pile of order slips and still not know which **categories** actually bring in money. The prices live in one list. The orders live in another. To answer the question, we have to join them, and we have to notice anything messy first.

### Our two tables
| File | What one row means | Key columns |
|---|---|---|
| `products.csv` | One product the shop sells | `product_id`, `product_name`, `category`, `unit_price`, `in_stock` |
| `orders.csv` | One order line (a product bought in one order) | `order_id`, `order_date`, `product_id`, `quantity`, `payment`, `cashier` |

The shared key is `product_id`. Revenue for a line is `quantity * unit_price`. That number does not exist until we merge.

These files are fictional teaching data. They are allowed to have gaps. That is the point.

### The analytical workflow (today's spine)
1. **Question** — what are we trying to learn?
2. **Load** — bring the tables into pandas.
3. **Inspect** — shape, types, missing values, strange values.
4. **Clean** — fix only what needs fixing, and say why.
5. **Transform** — add useful columns (for example a large-order flag, or line revenue after the merge).
6. **Merge** — join tables on a shared key; check the grain.
7. **Group / aggregate** — compare groups that answer the question.
8. **Plot** — a few charts that serve the question (not chart-for-chart's-sake).
9. **Conclude** — findings plus caveats.

Next session (Week 4 Session 2) goes deeper on choosing charts. Today we only need the workflow and two light plots.

## 1. Load the data

Import pandas (and matplotlib for two light charts later).

In [ ]:
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt

Load the product list.

In [ ]:
products = pd.read_csv("products.csv")

In [ ]:
products.head()

Load the orders.

In [ ]:
orders = pd.read_csv("orders.csv")

In [ ]:
orders.head()

If you get `FileNotFoundError`, the CSV is not in the same folder as this notebook. Move `products.csv` and `orders.csv` next to this notebook and run the load cells again.

`head()` shows the first 5 rows. `tail()` shows the last rows.

In [ ]:
products.tail()

In [ ]:
orders.tail()

### Your turn 1
Load only the first 5 rows of `products.csv` into a variable called `product_peek`, using `nrows=5`. Then display it.

In [ ]:
# SOLUTION
product_peek = pd.read_csv("products.csv", nrows=5)
product_peek

## 2. Inspect

Good analysis starts with looking. We check shape, columns, types, and a few summaries before we change anything.

In [ ]:
print("Products rows and columns:", products.shape)

In [ ]:
print("Orders rows and columns:", orders.shape)

In [ ]:
print(list(products.columns))

In [ ]:
print(list(orders.columns))

`info()` shows types and non-null counts. Scan for surprises.

In [ ]:
products.info()

In [ ]:
orders.info()

**What does one row mean (grain)?**
- In `products`, one row is one item on the shelf. There should be 12 products.
- In `orders`, one row is one line from an order. The same `order_id` should not appear twice unless we have a true duplicate row.

We will join them on `product_id`. That only works if the codes really match.

In [ ]:
products.describe()

In [ ]:
orders.describe()

How many products are in each category? How do payments break down? `value_counts()` answers those quickly. `dropna=False` keeps missing values visible.

In [ ]:
products["category"].value_counts()

In [ ]:
orders["payment"].value_counts(dropna=False)

In [ ]:
orders["cashier"].value_counts()

How many distinct product codes appear in each file? If orders has *more* distinct codes than the product list, some order lines may not match.

In [ ]:
print("Product codes in products:", products["product_id"].nunique())

In [ ]:
print("Product codes in orders:", orders["product_id"].nunique())

### Your turn 2
1. Print the unique category names in `products`.
2. How many distinct cashiers are in `orders`? Use `nunique()`.

In [ ]:
# SOLUTION
print(products["category"].unique())

In [ ]:
# SOLUTION
print(orders["cashier"].nunique())

## 3. Missing data, duplicates, and odd values

List the problems before you fix anything.

In [ ]:
products.isnull().sum()

In [ ]:
orders.isnull().sum()

There is a missing unit price in the product list, and missing quantities and payments in the orders. Let us see those rows.

In [ ]:
products[products["unit_price"].isnull()]

In [ ]:
orders[orders["quantity"].isnull()]

In [ ]:
orders[orders["payment"].isnull()]

**Duplicates.** `duplicated()` marks rows that are exact copies of an earlier row.

In [ ]:
print("Duplicate product rows:", products.duplicated().sum())

In [ ]:
print("Duplicate order rows:", orders.duplicated().sum())

In [ ]:
orders[orders.duplicated(keep=False)]

**Odd values.** A quantity of 0 is worth a look. It may be a typo, a cancelled line, or a real zero. We will decide after we see it.

In [ ]:
zero_qty = orders[orders["quantity"] == 0]

In [ ]:
zero_qty

**Problem list (write this in your notes):**
1. P12 Detergent has no unit price, so we cannot compute revenue for those orders yet.
2. Four orders have a missing quantity.
3. Three orders have a missing payment method (payment is not needed for revenue, but it is incomplete).
4. One order line is an exact duplicate.
5. One order has quantity 0.
6. Orders use 14 product codes, but the product list only has 12. Some keys will not match.

We will fix these in a working copy, and we will write a short justification for each change.

In [ ]:
orders_clean = orders.copy()

In [ ]:
products_clean = products.copy()

In [ ]:
print("Orders copy:", orders_clean.shape)

**Fix 1: drop the exact duplicate row.** It does not represent a second real sale.

In [ ]:
orders_clean = orders_clean.drop_duplicates()

In [ ]:
print("Orders after dropping the duplicate:", orders_clean.shape)

**Justification:** One row was an exact copy of another. Keeping it would double-count that sale.

We will handle missing quantities, the zero quantity, the missing price, and the unmatched keys after we merge, because some of those problems show up more clearly there.

## 4. Filter and transform

Boolean masks let us keep the rows we want. `.loc` lets us create or fill a column in a clear, step-by-step way.

First, look at one cashier's orders.

In [ ]:
bisi = orders_clean[orders_clean["cashier"] == "Bisi"]

In [ ]:
print("Bisi rows:", len(bisi))

In [ ]:
bisi.head()

Now flag larger orders. We start every row as `No`, then mark `Yes` where quantity is 4 or more.

In [ ]:
orders_clean["large_order"] = "No"

In [ ]:
orders_clean.loc[orders_clean["quantity"] >= 4, "large_order"] = "Yes"

In [ ]:
orders_clean["large_order"].value_counts(dropna=False)

Rows with a missing quantity stay as `No` because the condition is not True. That is fine for a flag, but those rows still cannot be used for revenue later.

### Your turn 3
From `products_clean`, keep only the Staples category. Store the result as `staples` and print how many rows it has.

In [ ]:
# SOLUTION
staples = products_clean[products_clean["category"] == "Staples"]
print("Staples products:", len(staples))

### Your turn 4
Using `orders_clean`, show the `order_id`, `product_id`, and `quantity` for rows where `large_order` is `"Yes"`.

In [ ]:
# SOLUTION
orders_clean.loc[orders_clean["large_order"] == "Yes", ["order_id", "product_id", "quantity"]]

## 5. Merge products with orders

We join the cleaned orders to the product list on `product_id`.

**Talking point for yourself:** before you merge, ask:
1. What does one row mean in each table?
2. Which column is the key?
3. Should this be one-to-one or many-to-one? (Many orders can point at one product.)
4. How many rows do I expect after the merge?

A **left** merge keeps every order line, and brings in product details where the key matches. Unmatched orders stay in the table with missing product columns. That is useful, because we want to *see* the bad keys.

In [ ]:
print("Orders before merge:", orders_clean.shape)

In [ ]:
print("Products before merge:", products_clean.shape)

In [ ]:
merged = orders_clean.merge(products_clean, on="product_id", how="left")

In [ ]:
print("Rows after merge:", merged.shape)

In [ ]:
merged.head()

Check for unmatched keys: order lines whose `product_name` is missing after the merge.

In [ ]:
unmatched = merged[merged["product_name"].isnull()]

In [ ]:
unmatched

In [ ]:
print("Unmatched order lines:", len(unmatched))

**What we found:** two order lines use product codes that are not in the product list (`P99` and `P0X`). They look like typing errors. We cannot invent a category or a price for them, so we will drop them from the revenue calculation and note them as a data quality issue.

Also check that we did not accidentally duplicate order lines.

In [ ]:
print("Duplicate order_id after merge:", merged.duplicated(subset=["order_id"]).sum())

### Your turn 5
Create `inner_merged` by merging `orders_clean` with `products_clean` on `product_id` using `how="inner"`. Print its shape. How does it compare with the 110-row left merge, and what disappeared?

In [ ]:
# SOLUTION
inner_merged = orders_clean.merge(products_clean, on="product_id", how="inner")
print(inner_merged.shape)

An inner merge quietly drops the unmatched orders. The left merge made them visible first. That is why we inspect keys before we choose the join type.

Now compute line revenue, then see how many rows are still missing a total.

In [ ]:
merged["line_total"] = merged["quantity"] * merged["unit_price"]

In [ ]:
print("Missing line_total:", merged["line_total"].isnull().sum())

In [ ]:
merged[merged["line_total"].isnull()][["order_id", "product_id", "quantity", "unit_price", "product_name", "line_total"]]

**Fix 2: keep a revenue table that only has usable line totals.**

We drop rows where `line_total` is missing. That removes:
- the two unmatched product codes,
- the four missing quantities,
- the Detergent orders (P12) that have no unit price,
- and the quantity-0 row, whose total would be 0 if it had a price (check whether it is in this list).

**Justification:** Revenue must be quantity times price. If either piece is missing, we should not guess. We report how many rows we set aside.

In [ ]:
revenue = merged.dropna(subset=["line_total"]).copy()

In [ ]:
print("Usable revenue rows:", revenue.shape)

In [ ]:
print("Rows set aside:", len(merged) - len(revenue))

## 6. Groupby: answer the mission

Compare total revenue, average line value, and order counts by category.

In [ ]:
by_category = revenue.groupby("category")["line_total"].agg(["sum", "mean", "count"])

In [ ]:
by_category = by_category.round(2)

In [ ]:
by_category = by_category.sort_values("sum", ascending=False)

In [ ]:
by_category

In [ ]:
print("Total revenue:", revenue["line_total"].sum())

**Read the table slowly.**
- Look at **count** first: is the group large enough to discuss?
- Staples will likely lead because rice and other staples are expensive *and* they sell.
- If Household has a very small count, do not treat it as a firm ranking.

Write one sentence in your notes before you plot anything.

### Your turn 6
Using `revenue`, group by `cashier` and find each cashier's total revenue (`sum`), average line (`mean`), and number of lines (`count`). Sort by total, highest first.

In [ ]:
# SOLUTION
by_cashier = revenue.groupby("cashier")["line_total"].agg(["sum", "mean", "count"])
by_cashier = by_cashier.round(2)
by_cashier = by_cashier.sort_values("sum", ascending=False)
by_cashier

## 7. Two light charts

Charts must answer a question. Today we only need two. Week 4 Session 2 will practise chart choice in more depth.

**Chart 1 question:** Which category brings in the most revenue?

In [ ]:
cat_totals = revenue.groupby("category")["line_total"].sum()

In [ ]:
cat_totals = cat_totals.sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(7, 4))

In [ ]:
plt.bar(cat_totals.index.astype(str), cat_totals.values)

In [ ]:
plt.title("Greenleaf Market: total revenue by category")

In [ ]:
plt.ylabel("Revenue")

In [ ]:
plt.tight_layout()

In [ ]:
plt.show()

**Chart 2 question:** How many usable order lines sit in each category? This stops us from reading a total without a count.

In [ ]:
cat_counts = revenue.groupby("category")["line_total"].count()

In [ ]:
cat_counts = cat_counts.reindex(cat_totals.index)

In [ ]:
plt.figure(figsize=(7, 4))

In [ ]:
plt.bar(cat_counts.index.astype(str), cat_counts.values)

In [ ]:
plt.title("Greenleaf Market: usable order lines by category")

In [ ]:
plt.ylabel("Number of order lines")

In [ ]:
plt.tight_layout()

In [ ]:
plt.show()

## 8. Findings and caveats

Use this template in your notes (and later in the homework paragraph):

1. **Finding:** Among usable order lines, Staples bring in most of the shop's revenue, far ahead of Drinks and Snacks.
2. **Support:** cite the groupby table (totals, averages, and counts) and the two charts.
3. **Caveat:** we set aside rows with missing quantity, missing price (Detergent), typo product codes, and a duplicate. Household, if it appears last, may rest on a very small number of rows. This dataset is fictional and covers two weeks.
4. **Decision:** before we tell the shopkeeper to "focus only on staples," we should fill in the Detergent price and confirm the typo codes.

Honesty is part of the workflow, not an afterthought.

## 9. Challenge (10 to 15 minutes)

Using the `revenue` table, find the **single product** (`product_name`) with the highest total revenue.

Report:
1. the product name,
2. its total revenue,
3. how many order lines that total is based on,
4. one caveat (for example: a high price, a short window, or rows you had to set aside).

In [ ]:
# SOLUTION
by_product = revenue.groupby("product_name")["line_total"].agg(["sum", "count"])
by_product = by_product.sort_values("sum", ascending=False)
by_product.head()

## Recap
- An analytical workflow starts with a **question**, not with a random step.
- `read_csv` loads tables; `shape`, `info`, `isnull`, and `duplicated` are the inspect habit.
- Boolean masks and `.loc` create flags and subsets without loops.
- `merge` needs a clear key and a grain check (row counts before and after, plus a look at unmatched keys).
- A left merge shows missing keys; an inner merge can hide them.
- `groupby` compares groups that answer the question. Always read the counts.
- Findings need **caveats**: missing prices, typo codes, and tiny groups can change the story.
- Next session (Week 4 Session 2) goes deeper on choosing charts that each answer a stated question.

## Homework
**Due:** before the next class. **Submit:** this notebook with Your turns completed, plus a short findings paragraph.

1. Finish **Your turns 1 to 6** and the **challenge**.
2. Write a short findings paragraph (about 8 to 12 lines) that answers the mission question for Greenleaf Market, includes at least one number from your tables, and ends with one clear caveat.
3. Skim the EDA project brief below and note which dataset idea you might use (the Greenleaf files are one option, not required).

## EDA PROJECT kickoff (starts this week)

This is the major Week 4 project. Structure:

**Problem -> Understanding -> Cleaning -> Exploration -> Viz -> Findings -> Conclusion**

**Deliverables**
- A notebook that follows the structure above and runs from top to bottom
- A cleaned dataset (CSV)
- A short README (dataset source, how to run, assumptions)
- A short findings report (at least 3 insights that go beyond describing a chart)

**Dataset choice is open.** Pick a question you care about and a public or class CSV you can load locally with `pd.read_csv`. The Greenleaf Market files from today are **one option**, not a requirement. Other ideas: a dataset you already used in this course, a beginner public table, or another small open-data CSV.

**Rules of the road**
- Start with a clear question.
- Inspect before you clean, and justify each fix.
- Use meaningful analysis (for example `groupby`), not charts alone.
- Every chart must answer a question.
- State caveats. Do not overclaim.

We will keep building this through Week 4 Session 2 (visualization and EDA). Come to the next class with a draft question and a candidate dataset name.